In [1]:
import os

# 1) Ensure we are in mlops
os.chdir(r"D:\labmentix\major\TravelGuru\travelguru-v5\backend\app")

# 2) Walk and print tree
for root, dirs, files in os.walk("."):
    level = root.count(os.sep)
    indent = "    " * level
    print(f"{indent}{os.path.basename(root)}/")
    sub_indent = "    " * (level + 1)
    for f in files:
        print(f"{sub_indent}{f}")


./
    app.py
    __init__.py
    adapters/
        interceptors.py
        __init__.py
    agents/
        __init__.py
        adk_remote_agent/
            agent_executor.py
            budget_and_experience_optimization_agent.py
            __init__.py
            __main__.py
        adk_travel_host/
            adk_tools.py
            remote_agent_connection.py
            travel_orchestrator_agent.py
            __init__.py
        crewai_remote_agent/
            agent_executor.py
            explorer_agent.py
            __init__.py
            __main__.py
        langgraph_agents/
            api.py
            composer_agent.py
            local_tool_router.py
            memory.py
            planner_agent.py
            travel_planner_graph.py
            __init__.py
            prompts/
                composer.txt
                composer_old.txt
                composer_old_backup.txt
                planner.txt
            shared/
                debug_a2a_methods.py
  

In [ ]:
import os, sys
import importlib
import pandas as pd

# 1) Ensure CWD is mlops root
if os.path.basename(os.getcwd()) == "notebook":
    os.chdir("..")
print("CWD:", os.getcwd())

# 2) Ensure mlops is importable (add backend/app to sys.path)
mlops_root = os.getcwd()                         # .../backend/app/mlops
backend_app_root = os.path.dirname(mlops_root)   # .../backend/app

if backend_app_root not in sys.path:
    sys.path.append(backend_app_root)

# 3) Import from mlops packages (NOT bare exception/utils)
from mlops.logging import setup_logging
from mlops.exception import dump_exception
from mlops.utils import db_utils

# 4) Optional: reload if you edit code often
importlib.reload(db_utils)

# 5) Setup logging once
setup_logging()


In [ ]:
from utils.db_utils import read_sql_to_df
# Safe DB pulls
try:
    flights_df = read_sql_to_df("""
    SELECT
        origin,
        destination,
        airline,
        dep_time,
        arr_time,
        duration_minutes,
        price,
        travel_date,
        cabin_class
    FROM database_ml.flights;
""")
    print("Flights OK:", len(flights_df))
except Exception as e:
    dump_exception(
        e, 
        context={"table": "flights"}, 
        data_sample=pd.DataFrame()  # Empty for now
    )
    raise  # Still fail fast

# Repeat for hotels_df, cabs_df
try:
    hotels_df = read_sql_to_df("""
    SELECT
        hotel_name,
        city,
        rating,
        price_per_night,
        amenities,
        star_category
    FROM database_ml.hotels;
""")
    print("Hotels OK:", len(hotels_df))
except Exception as e:
    dump_exception(e, context={"table": "hotels"})
    raise

try:
    cabs_df = read_sql_to_df("""
    SELECT
        ride_date,
        ride_time,
        pickup_location,
        drop_location,
        vehicle_type,
        distance_km,
        price,
        payment_method,
        driver_rating,
        customer_rating,
        ride_status
    FROM database_ml.cabs;
""")
    print("Cabs OK:", len(cabs_df))
except Exception as e:
    dump_exception(e, context={"table": "cabs"})
    raise


In [ ]:
print(flights_df.head())
print(hotels_df.head())
print(cabs_df.head())